# SegFusion - GPU Backend for Thesis Website
**Instructions:** Just click `Runtime > Run all` and wait. The last cell will print your Cloudflare URL.

1. Copy that URL
2. Paste it into your website's "Try It Yourself" section
3. Upload an image and watch the magic!

## Cell 1: Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn python-multipart albumentations timm ultralytics

## Cell 2: Mount Google Drive & Copy Model Weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# Change these paths if your models are in a different Drive folder
DRIVE_FOLDER = '/content/drive/MyDrive/bangladeshi_vehicle_segmentation_detection'

shutil.copy(os.path.join(DRIVE_FOLDER, 'best_hybrid.pt'), 'best_hybrid.pt')
shutil.copy(os.path.join(DRIVE_FOLDER, 'best_yolo.pt'), 'best_yolo.pt')

print('Hybrid size:', round(os.path.getsize('best_hybrid.pt') / 1e6, 1), 'MB')
print('YOLO size:', round(os.path.getsize('best_yolo.pt') / 1e6, 1), 'MB')
print('Models copied successfully!')

## Cell 3: Create the FastAPI Server Script

In [ ]:
%%writefile colab_fastapi_server.py
import os, time, cv2, torch, timm, uvicorn
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import Response, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from ultralytics import YOLO
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.colors as mcolors

# ---- Model Architecture ----
class CoordinateAttention(nn.Module):
    def __init__(self, in_channels, reduction=32):
        super().__init__()
        mip = max(8, in_channels // reduction)
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.conv1 = nn.Conv2d(in_channels, mip, 1)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.Hardswish(inplace=True)
        self.conv_h = nn.Conv2d(mip, in_channels, 1)
        self.conv_w = nn.Conv2d(mip, in_channels, 1)
    def forward(self, x):
        n, c, h, w = x.shape
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        x_h, x_w = torch.split(y, [h, w], dim=2)
        return x * torch.sigmoid(self.conv_h(x_h)) * torch.sigmoid(self.conv_w(x_w.permute(0, 1, 3, 2)))

class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch=256, rates=(6, 12, 18)):
        super().__init__()
        self.convs = nn.ModuleList([nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))])
        for r in rates:
            self.convs.append(nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=r, dilation=r, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True)))
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
        self.project = nn.Sequential(nn.Conv2d((len(rates) + 2) * out_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True), nn.Dropout(0.3))
    def forward(self, x):
        size = x.shape[-2:]
        res = [conv(x) for conv in self.convs]
        res.append(F.interpolate(self.pool(x), size=size, mode='bilinear', align_corners=False))
        return self.project(torch.cat(res, dim=1))

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, use_ca=True):
        super().__init__()
        self.ca = CoordinateAttention(skip_ch) if (skip_ch > 0 and use_ca) else None
        self.conv = nn.Sequential(nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True), nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        if skip is not None:
            if self.ca: skip = self.ca(skip)
            if skip.shape[-2:] != x.shape[-2:]: skip = F.interpolate(skip, size=x.shape[-2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class ProposedHybridASPPCA(nn.Module):
    def __init__(self, num_classes, pretrained=False):
        super().__init__()
        self.encoder = timm.create_model('resnet50', pretrained=pretrained, features_only=True, out_indices=(0,1,2,3,4))
        chs = self.encoder.feature_info.channels()
        self.aspp = ASPP(chs[4], 256)
        self.b0 = DecoderBlock(256, chs[3], 256)
        self.b1 = DecoderBlock(256, chs[2], 128)
        self.b2 = DecoderBlock(128, chs[1], 64)
        self.b3 = DecoderBlock(64, chs[0], 32)
        self.final = nn.Sequential(nn.Conv2d(32, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(True), nn.Conv2d(32, num_classes, 1))
    def forward(self, x):
        s = x.shape[-2:]
        f = self.encoder(x)
        d = self.b0(self.aspp(f[4]), f[3])
        d = self.b1(d, f[2])
        d = self.b2(d, f[1])
        d = self.b3(d, f[0])
        return self.final(F.interpolate(d, size=s, mode='bilinear', align_corners=False))

# ---- FastAPI App ----
app = FastAPI(title='SegFusion GPU Backend')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FP16 = torch.cuda.is_available()
NUM_CLASSES = 8
SIZE = 320

hybrid_model = yolo_model = transform = colors = probs_instance = None
stream_sem = stream_yolo = None

@app.on_event('startup')
async def load_models():
    global hybrid_model, yolo_model, transform, colors, stream_sem, stream_yolo
    print(f'Loading models to {DEVICE} (FP16={FP16})...')
    colors = np.zeros((NUM_CLASSES, 3), dtype=np.uint8)
    for i in range(1, NUM_CLASSES):
        colors[i] = (np.array(mcolors.hsv_to_rgb([(i-1)/max(1,NUM_CLASSES-1), 0.85, 0.95])) * 255).astype(np.uint8)
    hybrid_model = ProposedHybridASPPCA(NUM_CLASSES).to(DEVICE)
    hybrid_model.load_state_dict(torch.load('best_hybrid.pt', map_location=DEVICE))
    hybrid_model.eval()
    if FP16:
        hybrid_model.half()
        hybrid_model = torch.compile(hybrid_model, mode='reduce-overhead')
        stream_sem = torch.cuda.Stream()
        stream_yolo = torch.cuda.Stream()
    yolo_model = YOLO('best_yolo.pt')
    transform = A.Compose([A.Resize(SIZE, SIZE), A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])
    print('Backend Ready!')

def infer_frame(bgr):
    global probs_instance
    t0 = time.perf_counter()
    H, W = bgr.shape[:2]
    tensor = transform(image=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))['image'].unsqueeze(0).to(DEVICE, non_blocking=True)
    if FP16: tensor = tensor.half()
    if probs_instance is None or probs_instance.shape[1:] != (H, W):
        probs_instance = torch.zeros((NUM_CLASSES, H, W), dtype=torch.float16 if FP16 else torch.float32, device=DEVICE)
    probs_instance.zero_()
    probs_instance[0] = 0.5
    with torch.amp.autocast('cuda', enabled=FP16):
        if FP16:
            with torch.cuda.stream(stream_sem):
                sem = F.interpolate(F.softmax(hybrid_model(tensor), dim=1), size=(H,W), mode='bilinear', align_corners=False).squeeze(0)
            with torch.cuda.stream(stream_yolo):
                yr = next(yolo_model.predict(bgr, conf=0.35, imgsz=SIZE, device=DEVICE, verbose=False, stream=True))
                if yr.boxes is not None and yr.masks is not None and len(yr.boxes) > 0:
                    masks = F.interpolate(yr.masks.data.unsqueeze(1).half(), size=(H,W), mode='nearest').squeeze(1)
                    for i, box in enumerate(yr.boxes):
                        tc = int(box.cls[0].item()) + 1
                        if tc < NUM_CLASSES:
                            probs_instance[tc] = torch.maximum(probs_instance[tc], masks[i] * float(box.conf[0].item()))
            torch.cuda.synchronize()
        else:
            sem = F.interpolate(F.softmax(hybrid_model(tensor), dim=1), size=(H,W), mode='bilinear', align_corners=False).squeeze(0)
            yr = next(yolo_model.predict(bgr, conf=0.35, device=DEVICE, verbose=False, stream=True))
            if yr.boxes is not None and yr.masks is not None and len(yr.boxes) > 0:
                masks = F.interpolate(yr.masks.data.unsqueeze(1).float(), size=(H,W), mode='nearest').squeeze(1)
                for i, box in enumerate(yr.boxes):
                    tc = int(box.cls[0].item()) + 1
                    if tc < NUM_CLASSES:
                        probs_instance[tc] = torch.maximum(probs_instance[tc], masks[i] * float(box.conf[0].item()))
        mask = torch.argmax(0.55 * sem + 0.45 * probs_instance, dim=0).to(torch.uint8).cpu().numpy()
    if FP16: torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) * 1000
    overlay = cv2.addWeighted(bgr, 0.6, colors[mask], 0.4, 0)
    if yr.boxes is not None:
        for box in yr.boxes:
            x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
            cv2.rectangle(overlay, (x1,y1), (x2,y2), (0,255,128), 2)
            cv2.putText(overlay, f'{yr.names[int(box.cls[0].item())]} {float(box.conf[0].item()):.2f}', (x1,y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,128), 2)
    cv2.putText(overlay, f'Colab GPU: {ms:.1f}ms ({1000/max(0.001,ms):.1f} FPS)', (20,40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2, cv2.LINE_AA)
    return overlay

@app.post('/predict')
async def predict(file: UploadFile = File(...)):
    img = cv2.imdecode(np.frombuffer(await file.read(), np.uint8), cv2.IMREAD_COLOR)
    if img is None: return JSONResponse(status_code=400, content={'message': 'Invalid Image'})
    _, enc = cv2.imencode('.jpg', infer_frame(img))
    return Response(content=enc.tobytes(), media_type='image/jpeg')

if __name__ == '__main__':
    uvicorn.run(app, host='0.0.0.0', port=8000)

## Cell 4: Start Server + Cloudflare Tunnel
This cell will:
1. Download Cloudflare
2. Start the FastAPI server in background
3. Wait 15 seconds for models to load onto GPU
4. Open the Cloudflare tunnel

**Look for the `.trycloudflare.com` URL in the output and paste it into your website!**

In [ ]:
import subprocess, time

# Download cloudflared
subprocess.run(['wget', '-q', '-nc', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
subprocess.run(['chmod', '+x', 'cloudflared-linux-amd64'], check=True)

# Kill any old servers
subprocess.run(['pkill', '-f', 'uvicorn'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

# Start FastAPI server in background
server = subprocess.Popen(
    ['python', 'colab_fastapi_server.py'],
    stdout=open('server.log', 'w'),
    stderr=subprocess.STDOUT
)
print('Starting FastAPI server...')

# Wait for models to fully load
for i in range(20):
    time.sleep(1)
    with open('server.log', 'r') as f:
        log = f.read()
    if 'Backend Ready' in log:
        print('Models loaded on GPU!')
        break
    print(f'  Loading models... ({i+1}s)')
else:
    print('WARNING: Server may not be fully ready. Check server.log')
    print(open('server.log').read())

print()
print('='*60)
print('Starting Cloudflare Tunnel...')
print('Look for the .trycloudflare.com URL below!')
print('='*60)
print()

# Start cloudflare tunnel (this will block and keep running)
!./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8000